##### Setup


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS dbr_dev.yanquiel")
spark.sql(f"CREATE VOLUME IF NOT EXISTS dbr_dev.yanquiel.raw_data")


from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, IntegerType, DoubleType
import re

In [0]:
catalog = "dbr_dev"
schema = "yanquiel"
volume_path = f"/Volumes/{catalog}/{schema}/raw_data/world_population.csv"

In [0]:
population_schema = (
    StructType()
    .add("Rank", IntegerType())
    .add("CCA3", StringType())
    .add("Country/Territory", StringType())
    .add("Capital", StringType())
    .add("Continent", StringType())
    .add("2022 Population", IntegerType())
    .add("2020 Population", IntegerType())
    .add("2015 Population", IntegerType())
    .add("2010 Population", IntegerType())
    .add("2000 Population", IntegerType())
    .add("1990 Population", IntegerType())
    .add("1980 Population", IntegerType())
    .add("1970 Population", IntegerType())
    .add("Area (km²)", IntegerType())
    .add("Density (per km²)", DoubleType())
    .add("Growth Rate", DoubleType())
    .add("World Population Percentage", DoubleType())
    )

In [0]:
df_bronze = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(population_schema)
    .load(volume_path)
)

def clean_col_name(name):
    return re.sub(r"[^0-9a-zA-Z_]", "_", name).strip("_")

for old_name in df_bronze.columns:
    new_name = clean_col_name(old_name)
    if new_name != old_name:
        df_bronze = df_bronze.withColumnRenamed(old_name, new_name)

df_bronze.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.brz_world_population")


In [0]:
RENAME_MAP = {
    "Country_Territory": "country",
    "2022_Population": "population_2022",
    "Area__km": "area_km2",
    "Density__per_km": "density_km2",
    "Growth_Rate": "growth_rate",
    "World_Population_Percentage": "world_population_pct"
}

df_silver = spark.table(f"{catalog}.{schema}.brz_world_population")

for old_name, new_name in RENAME_MAP.items():
    df_silver = df_silver.withColumnRenamed(old_name, new_name)

for field in df_silver.schema.fields:
    if isinstance(field.dataType, T.StringType):
        df_silver = df_silver.withColumn(field.name, F.trim(F.col(field.name)))




df_silver = df_silver.select(
    "CCA3",
    "country",
    "Capital",
    "Continent",
    "population_2022",
    "area_km2",
    "density_km2",
    "growth_rate",
    "world_population_pct"
)


df_silver = df_silver.dropDuplicates()

df_silver = df_silver.dropna(subset=["country", "CCA3"])


df_silver.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.slv_world_population")


In [0]:
import requests

try:
    response = requests.get("https://countries.dev/countries", timeout=15)
    response.raise_for_status()
    data = response.json()
    print(f"OK - Countries received: {len(data)}")
except requests.exceptions.Timeout:
    print("Timeout: the API took longer than 15 seconds to respond")
except requests.exceptions.RequestException as e:
    print(f"Network error: {e}")

In [0]:
import pandas as pd

df_countries_api = spark.createDataFrame(pd.DataFrame(data))
df_countries_api.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.brz_countries_api")
df_countries_api.printSchema()

In [0]:
df_bronze_api = spark.table(f"{catalog}.{schema}.brz_countries_api")

df_silver_api = (
    df_bronze_api
    .select(
        "name",
        "alpha2Code",
        "alpha3Code",
        "region",
        "subregion",
        "capital",
        "population",
        "area",
        "populationDensity",
        F.col("currencies")[0]["code"].alias("currency_code"),
        F.col("currencies")[0]["name"].alias("currency_name")
    )
    .dropDuplicates()
    .dropna(subset=["alpha3Code"])
)

df_silver_api.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.slv_countries_api")
df_silver_api.printSchema()
print(f"Filas en Silver API: {df_silver_api.count()}")

In [0]:
df_silver_pop = spark.table(f"{catalog}.{schema}.slv_world_population")
df_silver_api = spark.table(f"{catalog}.{schema}.slv_countries_api")

df_gold_detail = (
    df_silver_pop.alias("s")
    .join(df_silver_api.alias("a"), F.col("s.CCA3") == F.col("a.alpha3Code"), "left")
    .select(
        "s.country",
        "s.Continent",
        "s.population_2022",
        "s.area_km2",
        "s.density_km2",
        "s.growth_rate",
        "a.capital",
        "a.currency_code",
        "a.region"
    )
)

df_gold_detail.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.gld_population_enriched")
df_gold_detail.printSchema()
print(f"Filas en Gold: {df_gold_detail.count()}")

In [0]:
df_gold_detail.filter(F.col("capital").isNull()).select("country").show(50, truncate=False)

In [0]:
df_gold_detail = spark.table(f"{catalog}.{schema}.gld_population_enriched")

df_gold_by_continent = (
    df_gold_detail.groupBy("Continent")
    .agg(
        F.sum("population_2022").alias("total_population"),
        F.avg("density_km2").alias("avg_density"),
        F.count("country").alias("num_countries"),
        F.avg("growth_rate").alias("avg_growth_rate")
    )
    .orderBy(F.desc("total_population"))
)

df_gold_by_continent.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.gld_population_by_continent")
df_gold_by_continent.display()